## **Download**

Import needed modules.

In [ ]:
import requests
import subprocess
import os

from tqdm.notebook import tqdm

Define some parameters.

In [ ]:
# time interval for the dataset
YEAR: int = 2025
MONTH: int = 7
DAY_FROM: int = 1   # first day considered
DAY_TO: int = 7     # last day considered

# download
SOURCE: str = "http://aisdata.ais.dk/"
ZIP_PATH: str = "dataset/raw/zip"
CSV_PATH: str = "dataset/raw/csv"

# upload
REPOSITORY_ID: str = "andsanv/ais-tracks"

Download raw, zip files from source.

In [ ]:
# make directory in case it does not exist
os.makedirs(ZIP_PATH, exist_ok=True)

# download all zips for each required day
for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Download"):
    # build url
    filename: str = f"aisdk-{YEAR}-{MONTH:02d}-{day:02d}.zip"
    url: str = SOURCE + filename

    filename = ZIP_PATH + '/' + filename

    # download file
    with requests.get(url, stream=True) as r:
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):  # 1 MB chunks
                if chunk:
                    f.write(chunk)

Unzip files into .csv format.

In [ ]:
# make directory in case it does not exist
os.makedirs(CSV_PATH, exist_ok=True)

# unzip each file into .csv format
for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Unzipping"):
    # define paths
    filename: str = f"aisdk-{YEAR}-{MONTH:02d}-{day:02d}"
    source: str = ZIP_PATH + '/' + filename
    destination: str = CSV_PATH + '/' + filename

    # unzip
    command = f"unzip -pq '{source}' > '{destination}.csv'"
    subprocess.run(command, shell=True, check=True)

## **Wrangling**

Import needed modules.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

Define parameters.

In [ ]:
# track filtering and segmentation
MIN_TRACK_LENGTH: int = 256         # minimum number of samples for a segment to be kept
MIN_SOG: int = 1                    # in knots
MAX_SOG: int = 50                   # in knots
MIN_TIMESPAN_TRACK: int = 3600      # two hours
MIN_DISPLACEMENT: int = 0.4         # equivalent to ~400 meters
TIME_GAP_SEGMENT: int = 1800        # 30 minutes

# grid limits and cell parameters
TOP_LEFT: tuple[int, int] = (0.0, 60.0)
BOTTOM_RIGHT: tuple[int, int] = (20.0, 50.0)
CELL_DIMENSIONS: tuple[int, int] = (0.01, 0.01)   # dimensions of the single cell in the grid

Define helpers.

In [ ]:
def clean_data_lazy(lazy_df):
    """
    Cleans a lazy dataframe in input removing null and uninteresting values.
    """

    lazy_df = lazy_df.filter(   # filter out uninteresting values
        pl.col("SOG").is_not_null() & pl.col("COG").is_not_null() &                         # remove null values
        (pl.col("Latitude") <= TOP_LEFT[1]) & (pl.col("Latitude") > BOTTOM_RIGHT[1]) &      # latitude must be inside grid
        (pl.col("Longitude") >= TOP_LEFT[0]) & (pl.col("Longitude") < BOTTOM_RIGHT[0]) &    # longitude must be inside grid
        (pl.col("Type of mobile").is_in(["Class A", "Class B"])) &                          # only interested in class A and B ship types
        (pl.col("MMSI").is_between(200_000_000, 775_999_999))                               # MMSI adheres to format
    )

    lazy_df = lazy_df.with_columns(pl.col("# Timestamp").str.to_datetime())     # cast timestamp into correct type
    lazy_df = lazy_df.unique(subset=["# Timestamp", "MMSI"], keep="first")      # remove duplicates

    return lazy_df


def filter_tracks_lazy(lazy_df):
    """
    Filters a lazy dataframe in input removing tracks which do not respect some metrics.
    """

    # compute aggregates per each MMSI
    group_stats = lazy_df.group_by("MMSI").agg([
        pl.len().alias("track_len"),
        pl.max("SOG").alias("max_SOG"),
        (pl.max("# Timestamp") - pl.min("# Timestamp")).dt.total_seconds().alias("track_timespan"),
        (
            (pl.max("Latitude") - pl.min("Latitude")).pow(2) + (pl.max("Longitude") - pl.min("Longitude")).pow(2)
        ).sqrt().alias("track_displacement")
    ])

    # select mmsi to keep based on the computed values
    valid_mmsi = group_stats.filter(
        (pl.col("track_len") >= MIN_TRACK_LENGTH) &             # filter out short (count) tracks
        (pl.col("max_SOG") >= MIN_SOG) &                        # filter out tracks containing slow ships
        (pl.col("max_SOG") <= MAX_SOG) &                        # filter out tracks containing very fast ships
        (pl.col("track_timespan") >= MIN_TIMESPAN_TRACK) &      # filter out short (time) tracks
        (pl.col("track_displacement") >= MIN_DISPLACEMENT)      # filter tracks concentrated in few cells
    ).select("MMSI")

    return lazy_df.join(valid_mmsi, on="MMSI", how="inner")     # apply the filter


def make_segments_lazy(lazy_df):
    """
    Takes a lazy dataframe in input and transforms tracks into segments.
    """

    # create segments
    lazy_df = lazy_df.sort(["MMSI", "# Timestamp"])     # sort based on time to be able to aggregate
    lazy_df = lazy_df.with_columns([                    # computes segment number for each sample considering the cumulative sum of time differences
        (
            (pl.col("# Timestamp").diff().dt.total_seconds().fill_null(0) >= TIME_GAP_SEGMENT)
            .cum_sum()
            .over("MMSI")
        ).alias("segment")
    ])

    # compute segment timespan
    segment_stats = lazy_df.group_by(["MMSI", "segment"]).agg([
        (pl.max("# Timestamp") - pl.min("# Timestamp")).dt.total_seconds().alias("segment_timespan")
    ])

    # keep only segments longer than a certain threshold
    valid_segments = segment_stats.filter(pl.col("segment_timespan") >= TIME_GAP_SEGMENT).select(["MMSI", "segment"])
    return lazy_df.join(valid_segments, on=["MMSI", "segment"], how="inner")


def transform_dataframe_lazy(lazy_df):
    """
    Applies some transformation to the attributes of a lazy dataframe in input.
    """

    lazy_df = lazy_df.rename({          # rename columns for major clarity
        "# Timestamp": "timestamp",
        "Type of mobile": "type",
        "Latitude": "y",
        "Longitude": "x"
    })

    lazy_df = lazy_df.with_columns([    # transform speed of the ship from knots to meters per second
        (pl.col("SOG") * 0.514444).cast(pl.Float32).alias("SOG"),
        pl.col("COG").cast(pl.Float32)
    ])

    lazy_df = lazy_df.with_columns([    # compute sine and cosine of the angle, to have continuity between 360° and 0°
        ((pl.col("COG") * np.pi / 180).sin()).alias("COG_sin"),
        ((pl.col("COG") * np.pi / 180).cos()).alias("COG_cos")
    ])

    lazy_df = lazy_df.with_columns([
        pl.col("x").cast(pl.Float32),   # downcast coordinates and MMSI
        pl.col("y").cast(pl.Float32),
        pl.col("MMSI").cast(pl.Int32),
        pl.col("type").cast(pl.Categorical)
    ])

    lazy_df = lazy_df.drop(["COG"])     # drop useless absolute value of the angle

    return lazy_df.select(["MMSI", "type", "timestamp", "segment", "x", "y", "SOG", "COG_sin", "COG_cos"])

Load .csvs and perform wrangling.

In [ ]:
# create a list to store dataframes
lazy_dfs: list = []

for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Cleaning"):
    # lazily load dataset
    lazy_df = pl.scan_csv(f"{CSV_PATH}/aisdk-{YEAR}-{MONTH:02d}-{day:02d}.csv")                                 # load
    lazy_df = lazy_df.select(["MMSI", "Type of mobile", "# Timestamp", "Longitude", "Latitude", "SOG", "COG"])  # select only needed cols

    # lazily transform the dataset
    lazy_df = clean_data_lazy(lazy_df)              # remove rows with nulls or uninteresting values
    lazy_df = filter_tracks_lazy(lazy_df)           # remove short tracks
    lazy_df = make_segments_lazy(lazy_df)           # divide tracks into segments
    lazy_df = transform_dataframe_lazy(lazy_df)     # transform some attributes

    lazy_dfs.append(lazy_df)                        # add dataframe to the list

Define some helpers for resampling.

In [ ]:
def interpolate_and_sample_segment(df: pl.DataFrame, delta: int = 30):
    """
    Interpolates raw segment and samples it to have a (fine) fixed time interval between samples.
    Returns a DataFrame with one sample every "delta" seconds.
    """
    # generate the target time stamps
    t_min = df["timestamp"].min()   # computes first and last timestamps of the segment
    t_max = df["timestamp"].max()

    grid = pl.DataFrame({           # creates a range object with one element every "delta" seconds
        "timestamp": pl.datetime_range(t_min, t_max, interval=f"{delta}s", eager=True)
    })

    # combine the time intervals with the original segment data
    cols = ["x", "y", "SOG", "COG_sin", "COG_cos"]  # columns where interpolation is needed

    combined = pl.concat([      # combines the old samples with new empty (attributes are set to null) samples
        df.select(["timestamp"] + cols),
        grid.with_columns([pl.lit(None).cast(pl.Float32).alias(c) for c in cols])
    ], how="diagonal")

    interpolated = (            # removes eventual duplicates, sorts and fills empty (null) attributes
        combined
        .unique(subset=["timestamp"], keep="first")
        .sort("timestamp")
        .with_columns([
            pl.col(c).interpolate() for c in cols
        ])
    )

    out = grid.join(interpolated, on="timestamp", how="inner")  # keep only samples on the "fixed" timestamps

    out = out.with_columns([    # restore missing attributes of the samples
        pl.lit(df["segment"][0]).alias("segment"),
        pl.lit(df["MMSI"][0]).alias("MMSI"),
        pl.lit(df["type"][0]).alias("type"),
    ])

    return out.select(["timestamp", "MMSI", "type", "segment", "x", "y", "SOG", "COG_sin", "COG_cos"])

Interpolate the dataset to solve gaps and discontinuities.

In [ ]:
# define output schema
output_schema = {
    "timestamp": pl.Datetime,
    "MMSI": pl.Int32,
    "type": pl.Categorical,
    "segment": pl.UInt32,
    "x": pl.Float32,
    "y": pl.Float32,
    "SOG": pl.Float32,
    "COG_sin": pl.Float32,
    "COG_cos": pl.Float32,
}

# resample each segment
for i in tqdm(range(DAY_TO + 1 - DAY_FROM), desc="Resampling"):
    lazy_dfs[i] = lazy_dfs[i].group_by(["MMSI", "segment"]) \
                              .map_groups(
                                  lambda group_df: interpolate_and_sample_segment(group_df),
                                  schema=output_schema
                              )

## **Upload**

Define needed modules.

In [ ]:
import huggingface_hub

Define some parameters.

In [ ]:
PARQUET_PATH: str = "dataset/processed/parquet" # for conversion
REPOSITORY_ID: str = "andsanv/ais-tracks"       # for upload

Convert to parquet format and save locally.

In [ ]:
# create target directory
os.makedirs(PARQUET_PATH, exist_ok=True)

# iterate over days
for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Converting"):
    # define variables
    filename: str = f"aisdk-{YEAR}-{MONTH:02d}-{day:02d}.parquet"
    destination: str = PARQUET_PATH + '/' + filename
    i: int = day - DAY_FROM

    # save parquet file
    lazy_dfs[i].select(list(output_schema.keys())).cast(output_schema).sink_parquet(destination)

Upload to the HuggingFace platform.

In [ ]:
# login to HuggingFace
huggingface_hub.login()

In [ ]:
# retrieve platform apis
api = huggingface_hub.HfApi()

# find files
files = sorted([f for f in os.listdir(PARQUET_PATH) if f.endswith(".parquet")])

for file_name in files:
    # define paths
    local_path = f"{PARQUET_PATH}/{file_name}"
    remote_path = file_name  # same name in the repo

    # upload
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=remote_path,
        repo_id=REPOSITORY_ID,
        repo_type="dataset"
    )